## Beágyazások

Az előző példánkban magas dimenziós bag-of-words vektorokon dolgoztunk, melyek hossza `vocab_size` volt, és kifejezetten alacsony dimenziós pozíciós reprezentációs vektorokat konvertáltunk ritka egy-hetes reprezentációvá. Ez az egy-hetes reprezentáció nem hatékony memóriahasználat szempontjából, ráadásul minden szót egymástól függetlenül kezelünk, azaz az egy-hetes kódolt vektorok nem fejezik ki a szavak közötti szemantikai hasonlóságot.

Ebben az egységben tovább vizsgáljuk a **News AG** adatállományt. Kezdésként töltsük be az adatokat, és szerezzünk be néhány definíciót az előző jegyzetfüzetből.


In [1]:
import torch
import torchtext
import numpy as np
from torchnlp import *
train_dataset, test_dataset, classes, vocab = load_dataset()
vocab_size = len(vocab)
print("Vocab size = ",vocab_size)

Loading dataset...


d:\WORK\ai-for-beginners\5-NLP\14-Embeddings\data\train.csv: 29.5MB [00:01, 18.8MB/s]                            
d:\WORK\ai-for-beginners\5-NLP\14-Embeddings\data\test.csv: 1.86MB [00:00, 11.2MB/s]                          


Building vocab...
Vocab size =  95812



## Mi az az embedding?

Az **embedding** ötlete az, hogy a szavakat alacsonyabb dimenziójú sűrű vektorokkal ábrázoljuk, amelyek valamilyen módon tükrözik a szó szemantikai jelentését. Később megvitatjuk, hogyan lehet értelmes szóembeddingeket létrehozni, de egyelőre csak gondoljunk az embeddingekre úgy, mint egy módra a szóvektor dimenziójának csökkentésére. 

Tehát egy embedding réteg egy szót vesz bemenetként, és egy megadott `embedding_size` méretű kimeneti vektort állít elő. Értelmezés szerint nagyon hasonló az `Linear` réteghez, de egyhetes kódolt vektor helyett egy szó sorszámát tudja bemenetként kezelni.

Ha az embedding réteget használjuk hálózatunk első rétegeként, átválthatunk a bag-of-words modellről a **beágyazott táska** modellre, ahol először minden szót a szövegünkben a hozzá tartozó embeddingre alakítunk, majd mindezeken az embeddingeken egy összegző függvényt számítunk ki, mint például `sum`, `average` vagy `max`.  

![Kép, amely egy beágyazott osztályozót mutat öt szóból álló sorozatra.](../../../../../translated_images/hu/embedding-classifier-example.b77f021a7ee67eee.webp)

Az osztályozó neurális hálózatunk egy embedding réteggel kezdődik, majd egy aggregációs réteggel, és tetején egy lineáris osztályozóval:


In [2]:
class EmbedClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super().__init__()
        self.embedding = torch.nn.Embedding(vocab_size, embed_dim)
        self.fc = torch.nn.Linear(embed_dim, num_class)

    def forward(self, x):
        x = self.embedding(x)
        x = torch.mean(x,dim=1)
        return self.fc(x)

### Változó hosszúságú szekvenciák kezelése

Ennek az architektúrának az eredményeként a hálózatunk számára minibatcheket kell létrehozni egy bizonyos módon. Az előző egységben, a bag-of-words használatakor, a minibatch összes BoW tenzora egyenlő méretű volt `vocab_size`, függetlenül a szövegünk tényleges hosszától. Amint áttérünk a szóbeágyazásokra, változó számú szó lesz minden szövegmintában, és amikor ezeket a mintákat minibatchekbe kombináljuk, paddinget kell alkalmaznunk.

Ezt ugyanazzal a technikával lehet elvégezni, hogy a datasource-nak `collate_fn` függvényt biztosítunk:


In [3]:
def padify(b):
    # b is the list of tuples of length batch_size
    #   - first element of a tuple = label, 
    #   - second = feature (text sequence)
    # build vectorized sequence
    v = [encode(x[1]) for x in b]
    # first, compute max length of a sequence in this minibatch
    l = max(map(len,v))
    return ( # tuple of two tensors - labels and features
        torch.LongTensor([t[0]-1 for t in b]),
        torch.stack([torch.nn.functional.pad(torch.tensor(t),(0,l-len(t)),mode='constant',value=0) for t in v])
    )

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=padify, shuffle=True)

### Beágyazásos osztályozó tanítása

Most, hogy definiáltuk a megfelelő adatbetöltőt, az előző egységben definiált tanítófüggvény segítségével megtaníthatjuk a modellt:


In [4]:
net = EmbedClassifier(vocab_size,32,len(classes)).to(device)
train_epoch(net,train_loader, lr=1, epoch_size=25000)

3200: acc=0.6415625
6400: acc=0.6865625
9600: acc=0.7103125
12800: acc=0.726953125
16000: acc=0.739375
19200: acc=0.75046875
22400: acc=0.7572321428571429


(0.889799795315499, 0.7623160588611644)

> **Megjegyzés**: Itt csak 25 ezer rekordra (kevesebb, mint egy teljes epochára) tanítunk az idő megtakarítása érdekében, de folytathatod a tanítást, írhatsz egy függvényt, amely több epochán keresztül tanít, és kísérletezhetsz a tanulási ráta paraméterrel a magasabb pontosság eléréséhez. Körülbelül 90%-os pontosságig el kell tudnod jutni.


### EmbeddingBag réteg és változó hosszúságú szekvencia reprezentációja

Az előző architektúrában minden szekvenciát azonos hosszúságúra kellett feltöltenünk, hogy egy minibatch-be illeszkedjenek. Ez nem a leghatékonyabb módja a változó hosszúságú szekvenciák reprezentálásának - egy másik megközelítés, ha egy **offset** vektort használunk, amely tartalmazza az összes, egy nagy vektorban tárolt szekvencia eltolásait.

![Kép egy offset szekvencia reprezentációról](../../../../../translated_images/hu/offset-sequence-representation.eb73fcefb29b46ee.webp)

> **Megjegyzés**: A fenti képen karakterek szekvenciáját mutatjuk, de a példánkban szavak szekvenciájával dolgozunk. Azonban a változó hosszúságú szekvenciák offset vektorral való reprezentálásának általános elve ugyanaz marad.

Az offset reprezentációval való munkához az [`EmbeddingBag`](https://pytorch.org/docs/stable/generated/torch.nn.EmbeddingBag.html) réteget használjuk. Ez hasonló az `Embedding`-hez, de tartalomvektort és offset vektort fogad bemenetként, valamint tartalmaz egy átlagoló réteget is, ami lehet `mean`, `sum` vagy `max`.

Íme a módosított hálózat, amely az `EmbeddingBag`-t használja:


In [5]:
class EmbedClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super().__init__()
        self.embedding = torch.nn.EmbeddingBag(vocab_size, embed_dim)
        self.fc = torch.nn.Linear(embed_dim, num_class)

    def forward(self, text, off):
        x = self.embedding(text, off)
        return self.fc(x)

Az adatállomány előkészítéséhez a tanításhoz meg kell adnunk egy átalakító függvényt, amely előkészíti az eltolásvektort:


In [6]:
def offsetify(b):
    # first, compute data tensor from all sequences
    x = [torch.tensor(encode(t[1])) for t in b]
    # now, compute the offsets by accumulating the tensor of sequence lengths
    o = [0] + [len(t) for t in x]
    o = torch.tensor(o[:-1]).cumsum(dim=0)
    return ( 
        torch.LongTensor([t[0]-1 for t in b]), # labels
        torch.cat(x), # text 
        o
    )

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=offsetify, shuffle=True)

Megjegyzendő, hogy ellentétben az összes korábbi példával, a hálózatunk most két paramétert fogad: adatvektort és eltolásvektort, amelyek különböző méretűek. Hasonlóképpen, az adatbetöltőnk is 3 értéket ad vissza a 2 helyett: mind a szöveg-, mind az eltolásvektorok jellemzőként kerülnek biztosításra. Ezért kissé módosítanunk kell az edzési függvényünket, hogy ezt kezelni tudja:


In [7]:
net = EmbedClassifier(vocab_size,32,len(classes)).to(device)

def train_epoch_emb(net,dataloader,lr=0.01,optimizer=None,loss_fn = torch.nn.CrossEntropyLoss(),epoch_size=None, report_freq=200):
    optimizer = optimizer or torch.optim.Adam(net.parameters(),lr=lr)
    loss_fn = loss_fn.to(device)
    net.train()
    total_loss,acc,count,i = 0,0,0,0
    for labels,text,off in dataloader:
        optimizer.zero_grad()
        labels,text,off = labels.to(device), text.to(device), off.to(device)
        out = net(text, off)
        loss = loss_fn(out,labels) #cross_entropy(out,labels)
        loss.backward()
        optimizer.step()
        total_loss+=loss
        _,predicted = torch.max(out,1)
        acc+=(predicted==labels).sum()
        count+=len(labels)
        i+=1
        if i%report_freq==0:
            print(f"{count}: acc={acc.item()/count}")
        if epoch_size and count>epoch_size:
            break
    return total_loss.item()/count, acc.item()/count


train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6153125
6400: acc=0.6615625
9600: acc=0.6932291666666667
12800: acc=0.715078125
16000: acc=0.7270625
19200: acc=0.7382291666666667
22400: acc=0.7486160714285715


(22.771553103007037, 0.7551983365323096)


## Szemantikus beágyazások: Word2Vec

Előző példánkban a modell beágyazási rétege megtanulta szavakat vektoros reprezentációra leképezni, azonban ez a reprezentáció nem hordozott túl sok szemantikai jelentést. Jó lenne olyan vektoros reprezentációt tanulni, ahol a hasonló szavak vagy szinonimák olyan vektoroknak felelnek meg, amelyek vektor-távolság szempontjából közel vannak egymáshoz (pl. euklideszi távolság).

Ehhez előre kell tanítanunk a beágyazási modellünket egy nagy szöveggyűjteményen egy speciális módon. Az egyik első módszer a szemantikus beágyazások tanítására a [Word2Vec](https://en.wikipedia.org/wiki/Word2vec). Ez két fő architektúrán alapul, amelyeket a szavak elosztott reprezentációjának előállítására használnak:

 - **Folyamatos szótáska (Continuous bag-of-words, CBoW)** — ebben az architektúrában a modellt arra tanítjuk, hogy egy szót megjósoljon a környező kontextusból. Adva az n-gramot $(W_{-2},W_{-1},W_0,W_1,W_2)$, a modell célja $W_0$ megjóslása $(W_{-2},W_{-1},W_1,W_2)$ alapján.
 - **Folyamatos skip-gram** ellentétes a CBoW-val. A modell a kontextusban lévő szavak ablakát használja a jelenlegi szó megjóslására.

A CBoW gyorsabb, míg a skip-gram lassabb, de jobb munkát végez a ritkább szavak reprezentálásában.

![Kép, amely a CBoW és Skip-Gram algoritmusokat mutatja be a szavak vektorokká alakításához.](../../../../../translated_images/hu/example-algorithms-for-converting-words-to-vectors.fbe9207a726922f6.webp)

A Google News adathalmazon előre betanított word2vec beágyazással kísérletezhetünk a **gensim** könyvtár használatával. Alább megtaláljuk a "neural" szóhoz leginkább hasonló szavakat.

> **Megjegyzés:** Amikor először hozol létre szóvektorokat, azok letöltése eltarthat egy ideig!


In [8]:
import gensim.downloader as api
w2v = api.load('word2vec-google-news-300')

In [9]:
for w,p in w2v.most_similar('neural'):
    print(f"{w} -> {p}")

neuronal -> 0.7804799675941467
neurons -> 0.7326500415802002
neural_circuits -> 0.7252851724624634
neuron -> 0.7174385190010071
cortical -> 0.6941086649894714
brain_circuitry -> 0.6923246383666992
synaptic -> 0.6699118614196777
neural_circuitry -> 0.6638563275337219
neurochemical -> 0.6555314064025879
neuronal_activity -> 0.6531826257705688


Számíthatunk továbbá vektorembeddingeket is a szóból, amelyeket osztályozási modell tanításához használunk (az átláthatóság kedvéért csak az első 20 komponensét mutatjuk a vektornak):


In [10]:
w2v.word_vec('play')[:20]

array([ 0.01226807,  0.06225586,  0.10693359,  0.05810547,  0.23828125,
        0.03686523,  0.05151367, -0.20703125,  0.01989746,  0.10058594,
       -0.03759766, -0.1015625 , -0.15820312, -0.08105469, -0.0390625 ,
       -0.05053711,  0.16015625,  0.2578125 ,  0.10058594, -0.25976562],
      dtype=float32)

A szemantikai beágyazások nagyszerűsége, hogy manipulálhatjuk a vektor-kódolást a szemantika megváltoztatásához. Például megkérhetjük, hogy találjon egy olyan szót, amelynek vektorábrázolása a lehető legközelebb van a *king* és *woman* szavakhoz, és a *man* szótól pedig minél távolabb:


In [10]:
w2v.most_similar(positive=['king','woman'],negative=['man'])[0]

('queen', 0.7118192911148071)

Mind a CBoW, mind a Skip-Grams „prediktív” beágyazások, mivel csak a helyi kontextusokat veszik figyelembe. A Word2Vec nem használja ki a globális kontextus előnyeit.

A **FastText** a Word2Vec-re épít azzal, hogy minden szóhoz és a szóban található karakter n-grammokhoz vektorképviseletet tanul. A képviseletek értékeit minden tanítási lépésnél egyetlen vektorba átlagolják. Bár ez sokkal nagyobb számítási igényt jelent az előtanítás során, a szóbeágyazások számára lehetővé teszi, hogy szavakon belüli (al-szó) információkat is kódoljanak.

Egy másik módszer, a **GloVe**, a ko-prevalencia mátrix ötletét használja, és neurális módszerekkel bontja le a ko-prevalencia mátrixot kifejezőbb és nemlineáris szótvektorokká.

Játszhatsz a példával, ha a beágyazásokat FastText-re és GloVe-re cseréled, mivel a gensim több különböző szóbeágyazási modellt támogat.


## Előre betanított beágyazások használata PyTorch-ban

Az előző példát módosíthatjuk úgy, hogy az embedding rétegünk mátrixát előre betöltött szemantikai beágyazásokkal, például Word2Vec-szel töltsük fel. Figyelembe kell vennünk, hogy az előre betanított beágyazás és a szövegkorpusunk szókincse valószínűleg nem fog egyezni, ezért a hiányzó szavakhoz véletlenszerű értékekkel inicializáljuk a súlyokat:


In [11]:
embed_size = len(w2v.get_vector('hello'))
print(f'Embedding size: {embed_size}')

net = EmbedClassifier(vocab_size,embed_size,len(classes))

print('Populating matrix, this will take some time...',end='')
found, not_found = 0,0
for i,w in enumerate(vocab.get_itos()):
    try:
        net.embedding.weight[i].data = torch.tensor(w2v.get_vector(w))
        found+=1
    except:
        net.embedding.weight[i].data = torch.normal(0.0,1.0,(embed_size,))
        not_found+=1

print(f"Done, found {found} words, {not_found} words missing")
net = net.to(device)

Embedding size: 300
Populating matrix, this will take some time...Done, found 41080 words, 54732 words missing


Most tanítsuk meg a modellünket. Vegyük észre, hogy a modell tanításának ideje jelentősen nagyobb, mint az előző példában, a nagyobb beágyazási réteg mérete miatt, és így jóval több paraméter miatt. Emellett ezért előfordulhat, hogy több példán kell tanítanunk a modellt, ha el akarjuk kerülni a túlillesztést.


In [12]:
train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6359375
6400: acc=0.68109375
9600: acc=0.7067708333333333
12800: acc=0.723671875
16000: acc=0.73625
19200: acc=0.7463541666666667
22400: acc=0.7560714285714286


(214.1013875559821, 0.7626759436980166)

Esetünkben nem látunk óriási pontosságnövekedést, ami valószínűleg a teljesen eltérő szókincsnek köszönhető. 
A különböző szókincsek problémájának leküzdésére az alábbi megoldások egyikét alkalmazhatjuk:
* A word2vec modell újratanítása a saját szókincsünkre
* Az adatainkat a előre betanított word2vec modell szókincsével betölteni. Az adatbetöltés során megadható a betöltéshez használt szókincs.

Az utóbbi megközelítés tűnik könnyebbnek, különösen azért, mert a PyTorch `torchtext` keretrendszer beépített támogatást tartalmaz az embeddingekhez. Például a következő módon példányosíthatunk GloVe alapú szókincset:


In [14]:
vocab = torchtext.vocab.GloVe(name='6B', dim=50)

100%|█████████▉| 399999/400000 [00:15<00:00, 25411.14it/s]


A betöltött szókincs a következő alapműveletekkel rendelkezik:
* `vocab.stoi` szótár lehetővé teszi, hogy a szót a szótári indexére konvertáljuk
* `vocab.itos` az ellenkezőjét teszi - számmá alakítja a szót
* `vocab.vectors` az embedding vektorok tömbje, tehát egy szó embeddingjének megszerzéséhez `s` használni kell `vocab.vectors[vocab.stoi[s]]`

Íme egy példa az embeddingek manipulálására, hogy bemutassuk az egyenletet: **kind-man+woman = queen** (kicsit módosítanom kellett a szorzótényezőt, hogy működjön):


In [15]:
# get the vector corresponding to kind-man+woman
qvec = vocab.vectors[vocab.stoi['king']]-vocab.vectors[vocab.stoi['man']]+1.3*vocab.vectors[vocab.stoi['woman']]
# find the index of the closest embedding vector 
d = torch.sum((vocab.vectors-qvec)**2,dim=1)
min_idx = torch.argmin(d)
# find the corresponding word
vocab.itos[min_idx]

'queen'

Ahhoz, hogy a klasszifikátort ezeknek a beágyazásoknak a felhasználásával képezzük, először a GloVe szókincs segítségével kell kódolnunk az adatállományunkat:


In [16]:
def offsetify(b):
    # first, compute data tensor from all sequences
    x = [torch.tensor(encode(t[1],voc=vocab)) for t in b] # pass the instance of vocab to encode function!
    # now, compute the offsets by accumulating the tensor of sequence lengths
    o = [0] + [len(t) for t in x]
    o = torch.tensor(o[:-1]).cumsum(dim=0)
    return ( 
        torch.LongTensor([t[0]-1 for t in b]), # labels
        torch.cat(x), # text 
        o
    )

Ahogy fent láttuk, minden vektoros beágyazás a `vocab.vectors` mátrixban tárolódik. Ez nagyon megkönnyíti ezen súlyok beolvasását az embedding réteg súlyai közé egyszerű másolással:


In [17]:
net = EmbedClassifier(len(vocab),len(vocab.vectors[0]),len(classes))
net.embedding.weight.data = vocab.vectors
net = net.to(device)

Most tanítsuk be a modellünket, és nézzük meg, hogy jobb eredményeket kapunk-e:


In [18]:
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=offsetify, shuffle=True)
train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6271875
6400: acc=0.68078125
9600: acc=0.7030208333333333
12800: acc=0.71984375
16000: acc=0.7346875
19200: acc=0.7455729166666667
22400: acc=0.7529464285714286


(35.53972978646833, 0.7575175943698017)

Az egyik oka annak, hogy nem látunk jelentős pontosságnövekedést, az, hogy az adatbázisunkból származó néhány szó hiányzik az előre betanított GloVe szótárból, így gyakorlatilag figyelmen kívül vannak hagyva. Ennek leküzdésére az adatbázisunkon saját beágyazásokat taníthatunk. 


## Kontextuális Beágyazások

A hagyományos, előre betanított beágyazási reprezentációk, mint például a Word2Vec egyik kulcsfontosságú korlátja a szóértelem-elkülönítés problémája. Bár az előre betanított beágyazások képesek megragadni a szavak jelentésének egy részét a kontextusban, egy szó minden lehetséges jelentését ugyanabba a beágyazásba kódolják. Ez problémákat okozhat a további modellekben, mivel sok szó, például a 'play' különböző jelentéssel bír a használat kontextusától függően.

Például a 'play' szó az alábbi két különböző mondatban egészen más jelentéssel bír:
- Elmentem egy **előadásra** a színházban.
- John a barátaival akar **játszani**.

A fenti előre betanított beágyazások mindkét jelentést egyszerre jelenítik meg a 'play' szó esetében ugyanabban a beágyazásban. Ennek a korlátozásnak a leküzdéséhez olyan beágyazásokat kell építenünk, amelyek a **nyelvi modellen** alapulnak, amelyet egy nagy szövegkorpuszra tanítottak, és *tudja*, hogyan illeszthetők össze a szavak különböző kontextusokban. A kontextuális beágyazások tárgyalása e tananyag hatókörén kívül esik, de visszatérünk rájuk, amikor a nyelvi modellekről beszélünk a következő egységben.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Jogi nyilatkozat**:
Ez a dokumentum az AI fordítási szolgáltatás, a [Co-op Translator](https://github.com/Azure/co-op-translator) segítségével készült. Bár az pontosságra törekszünk, kérjük, vegye figyelembe, hogy az automatikus fordítások hibákat vagy pontatlanságokat tartalmazhatnak. Az eredeti dokumentum az anyanyelvén tekintendő hiteles forrásnak. Fontos információk esetén professzionális emberi fordítást javasolunk. Nem vállalunk felelősséget semmilyen félreértésért vagy téves értelmezésért, amely ebből a fordításból ered.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
